# 04 · Features + authenticity branch (train, evaluate, audits)

**Accelerator:** GPU **T4 x2** · **Attach:** `abo-prepared`, every `abo-fakes-*` you have, `cf-subset`, optional `phone-set`, `univfd-weights` · **Output dataset name:** `lv-features-checkpoints`

Index → nuisance-only shortcut audit (must be near 0.5) → CLIP + log-DCT features (canonical for all; JPEG 90/70/50/30
for evaluation splits; 70/50 for the train split as augmentation) → tags:

* `main`: SDXL-Turbo + SD 1.5 (what exists), compression augmentation; Kandinsky unseen.
* `main_noaug`, `turbo_only`, `control_mismatch` (CF-trained confound check), `all3` (adds Kandinsky; a separate, predeclared experiment; lower priority).

Validation is split by family into a selection half and a calibration half; thresholds are frozen there and
reported as actual TPR/FPR on test. Then: JPEG curve on identical images, per-generator rows, UnivFD baseline
if present, PNG storage diagnostic. On the **pilot**, only `main` is trained.

## 0. Setup (same cell in every notebook)

**Before running:** attach the code as a Kaggle Dataset named `listing-verifier-code`
(upload the repo folder without `data/`, `outputs/`, `web/node_modules/`), **or** set `REPO_URL`
to your GitHub repo. Turn on **Internet** (pip, ABO and HF downloads). For the GPU notebooks pick
**Settings → Accelerator → GPU T4 x2**. Store your Hugging Face token as the Kaggle secret `HF_TOKEN`.

In [ ]:
import os, shutil, subprocess, sys, pathlib, json
REPO_URL = "https://github.com/kvamsi-iisc/DLVA-Final-Project-Forensic-Verification-of-Marketplace-Listings.git"   # used if the code dataset is not attached
WORK = pathlib.Path("/kaggle/working"); REPO = WORK / "repo"; DATA = WORK / "data"; OUT = WORK / "outputs"
code_ds = pathlib.Path("/kaggle/input/listing-verifier-code")
if not REPO.exists():
    if code_ds.exists():
        shutil.copytree(code_ds, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
os.environ["LV_DATA_ROOT"] = str(DATA); os.environ["LV_OUTPUT_ROOT"] = str(OUT)
os.environ["HF_HOME"] = str(WORK / "hf_cache")           # model weights land here (removed before publishing)
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret:", type(e).__name__, "(fine unless a gated model is needed)")
print("python", sys.version.split()[0])
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], capture_output=True, text=True)
if pip.returncode != 0:
    print(pip.stdout[-3000:], pip.stderr[-3000:])          # show WHY before stopping (e.g. a pin without a wheel for this Python)
    raise SystemExit("pip install failed; see output above")
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

LOGS = OUT / "logs"; LOGS.mkdir(parents=True, exist_ok=True)

def run(*args):
    """Run a pipeline script; stream its output to the cell AND to OUT/logs/<script>_<time>.log,
    append a completion record to OUT/logs/completed.jsonl, and stop the notebook on a non-zero exit."""
    import time, datetime
    cmd = [sys.executable, *[str(a) for a in args]]
    name = pathlib.Path(str(args[0])).stem
    stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    log_path = LOGS / f"{name}_{stamp}.log"
    print(">>>", " ".join(cmd[1:]), "->", log_path.name, flush=True)
    t0 = time.time()
    with open(log_path, "w") as log:
        log.write("$ " + " ".join(cmd) + "\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True); log.write(line)
        rc = proc.wait()
    record = {"script": name, "args": cmd[2:], "returncode": rc, "seconds": round(time.time() - t0, 1),
              "log": str(log_path), "finished_utc": datetime.datetime.utcnow().isoformat() + "Z"}
    with open(LOGS / "completed.jsonl", "a") as f:
        f.write(json.dumps(record) + "\n")
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

def attach(slug, sub, dest, must_contain=None):
    """Copy a sub-folder of an attached Kaggle dataset into DATA/OUT (copy, not symlink: scripts write next to inputs).
    Refuses silently stale copies: if `must_contain` is missing from an existing dest, the copy is redone."""
    base = pathlib.Path("/kaggle/input") / slug
    src = base / sub if (base / sub).exists() else base
    dst = pathlib.Path(dest)
    if dst.exists() and (must_contain is None or (dst / must_contain).exists()):
        print("exists:", dst); return
    if dst.exists():
        shutil.rmtree(dst)
    if not src.exists():
        raise FileNotFoundError(f"dataset {slug} is not attached (expected {src})")
    print("copy", src, "->", dst); shutil.copytree(src, dst)

run("scripts/env_report.py", "--out", OUT / "results" / "env_report.json")
import torch; print(f"visible GPUs: {torch.cuda.device_count()}")


In [ ]:
FAKE_SLUGS = ['abo-fakes-pilot', 'abo-fakes-turbo-t2i', 'abo-fakes-turbo-i2i', 'abo-fakes-sd15-i2i', 'abo-fakes-kand-t2i', 'abo-fakes-kand-i2i']
attach("abo-prepared", "data/processed", DATA / "processed", must_contain="abo_clean.csv")
for slug in FAKE_SLUGS:
    src = pathlib.Path("/kaggle/input") / slug
    if not src.exists():
        print("not attached:", slug); continue
    fakes_root = next(p for p in [src / "data" / "fakes", src / "fakes", src] if p.exists())
    shutil.copytree(fakes_root, DATA / "fakes", dirs_exist_ok=True); print("merged", slug)
print("fake chunks:", len(list((DATA / "fakes").rglob("chunk_*.csv"))))
if pathlib.Path("/kaggle/input/cf-subset").exists(): attach("cf-subset", "data/forensics", DATA / "forensics", must_contain="forensics_subset.csv")
if pathlib.Path("/kaggle/input/phone-set").exists(): attach("phone-set", "phone", DATA / "phone", must_contain="labels.csv")
uf = pathlib.Path("/kaggle/input/univfd-weights/fc_weights.pth")
if uf.exists():
    (OUT / "checkpoints").mkdir(parents=True, exist_ok=True); shutil.copy(uf, OUT / "checkpoints" / "univfd_fc_weights.pth")
PILOT = not pathlib.Path("/kaggle/input/abo-fakes-turbo-t2i").exists()   # pilot = only the small run is attached
print("PILOT mode:", PILOT)

In [ ]:
run("scripts/05_build_index.py")
audit = json.load(open(DATA / "processed" / "nuisance_audit.json")); print(json.dumps(audit, indent=1)[:1500])
assert audit.get("nuisance_only_test_auroc", 0.5) < 0.6, "geometry/storage leaks the label: fix the data before training"

In [ ]:
run("scripts/06_extract_features.py", "--qualities", "0", "--gpus", "auto")
run("scripts/06_extract_features.py", "--qualities", "90,70,50,30", "--splits", "test,phone,forensics", "--gpus", "auto")
run("scripts/06_extract_features.py", "--qualities", "70,50", "--splits", "train", "--gpus", "auto")

In [ ]:
# The pilot has SDXL-Turbo fakes only, so its `main` is trained on that generator; the full run uses the default (turbo + sd15).
# Every requested generator must exist in the index (exit 3 otherwise): if you skipped sd15, pass --train-generators sdxl-turbo here.
tags = [("main", ["--train-generators", "sdxl-turbo"])] if PILOT else [("main", []), ("main_noaug", ["--augment-qualities", ""]), ("turbo_only", ["--train-generators", "sdxl-turbo"]),
                                     ("control_mismatch", ["--train-fake-domain", "forensics"]), ("all3", ["--train-generators", "sdxl-turbo,sd15,kandinsky22"])]
for tag, extra in tags:
    try:
        run("scripts/07_train_authenticity.py", "--tag", tag, *extra); run("scripts/08_eval_authenticity.py", "--tag", tag)
    except subprocess.CalledProcessError as e:
        if tag != "main" and e.returncode == 3:      # exit 3 = 'no data for this configuration' (e.g. no Kandinsky attached)
            print(f"[skip] {tag}: no data for this configuration"); continue
        raise                                          # OOM, broken training etc. must stop the notebook
try: run("scripts/14_png_diagnostic.py", "--tag", "main", "--split", "test")
except subprocess.CalledProcessError as e:
    if e.returncode != 3: raise
    print("[skip] png diagnostic: no PNG twins in this run")

### Read the results like an examiner
* `train_summary.json → recommended_mode`: if fusion does not beat the probe on the calibration half, the demo runs in probe mode.
* `catalogue/all_seen`: seen generators. Read `tpr_at_threshold` / `fpr_at_threshold` (frozen threshold), not only AUROC.
* `catalogue/kandinsky22 [unseen_generator_same_domain]` under `main`: the in-domain transfer number; `catalogue/sd15` under `turbo_only` is a second one.
* `forensics/internal`: unseen generators AND unseen domain, on the same reserved 40% for every tag.
* `jpeg_curve.png` for `main` vs `main_noaug`: the measured robustness story, whatever it turns out to be.
* `control_mismatch`: near chance on `catalogue/*` while high on `forensics/internal` means the style confound is real.
* `png_diagnostic.json`: the storage cost of the frequency cue.

In [ ]:
import pandas as pd
from IPython.display import Image as IImage, display
for d in sorted((OUT / "results").glob("*/summary.csv")):
    s = pd.read_csv(d); tag = d.parent.name
    print(f"\n=== {tag} ==="); print(s[s["quality"] == 0].pivot_table(index="test_set", columns="model", values=["auroc", "tpr_at_threshold", "fpr_at_threshold"]).round(3).to_string())
    if (d.parent / "jpeg_curve.png").exists(): display(IImage(str(d.parent / "jpeg_curve.png")))
print(json.load(open(OUT / "checkpoints" / "main" / "train_summary.json"))["recommended_mode"])

## Persisting outputs
When this notebook finishes under **Save Version → Save & Run All (Commit)**, everything in `/kaggle/working`
except the model cache is kept as the notebook output. Then: *Output tab → "New Dataset"* and name it as stated
at the top of this notebook. The next notebook attaches that dataset. The cell below checks the artifacts this
notebook promised actually exist, then removes the model cache so the output stays small.

In [ ]:
required = ["data/features/features_q0.npz", "outputs/checkpoints/main/probe.npz", "outputs/results/main/summary.csv", "data/processed/index.csv"]
run("scripts/verify_outputs.py", *[WORK / r for r in required])   # exits non-zero on a missing/empty artifact
shutil.rmtree(WORK / "hf_cache", ignore_errors=True)
shutil.rmtree(REPO / "web" / "node_modules", ignore_errors=True)
print(subprocess.run(["du", "-sh", str(DATA), str(OUT)], capture_output=True, text=True).stdout)